# TESSERA embeddings — Coimbra municipality pilot (2024)

**Purpose.** Explore the satellite/EO foundation-model data on its own:
tile coverage, embedding shape/dtype, and whether the values look spatially
coherent. Split out of
`01_problem_exploration/available_data_exploration.ipynb`, per
`notebooks/README.md`.

**Scope.** Exploratory only. Pilot download for access validation (see
`data/README.md`): 12 tiles, year 2024 only, over the Coimbra
**municipality** boundary — a different, smaller area than the
iNaturalist pilot in `../02_citizen_science/` (which used the Coimbra
district). See `../01_problem_exploration/` for the overall file
inventory, and `../04_data_integration/` for the first
feasibility check linking this to the citizen-science observations.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from geotessera import dequantize_embedding

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
TESSERA_DIR = RAW / "satellite" / "tessera_coimbra_pilot"
TESSERA_YEAR = 2024
TIFF_DIR = TESSERA_DIR / "global_0.1_degree_tiff_all"
tile_dirs = sorted((TESSERA_DIR / "global_0.1_degree_representation" / str(TESSERA_YEAR)).iterdir())

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110

## 1. Tile inventory

Per `data/README.md`: 12 tiles, quantised `int8` embeddings (128 channels) +
per-pixel scale factors, with a georeferenced landmask GeoTIFF per tile. The
`.npy` files carry no georeference — the matching GeoTIFF provides the CRS
and transform.

In [ ]:
TIFF_DIR = TESSERA_DIR / "global_0.1_degree_tiff_all"
tile_rows = []
for d in tile_dirs:
    emb = np.load(d / f"{d.name}.npy", mmap_mode="r")
    scales = np.load(d / f"{d.name}_scales.npy", mmap_mode="r")
    with rasterio.open(TIFF_DIR / f"{d.name}.tiff") as src:
        mask = src.read(1)
        tile_rows.append({
            "tile": d.name, "emb_shape": emb.shape, "emb_dtype": str(emb.dtype),
            "scales_shape": scales.shape, "tiff_shape": src.shape, "crs": src.crs.to_string(),
            "pixel_m": src.res[0], "shapes_match": emb.shape[:2] == src.shape,
            "pct_land": round((mask == 1).mean() * 100, 1),
        })
tessera_tiles = pd.DataFrame(tile_rows)
tessera_tiles

Dequantisation follows `geotessera.dequantize_embedding` (embedding × per-pixel
scale). Below: value range and a false-colour view of one tile (first 3
principal components of the 128-d embedding, fitted on a pixel sample) to
check the data look spatially coherent.

In [ ]:
demo = tile_dirs[len(tile_dirs) // 2]
emb = dequantize_embedding(np.load(demo / f"{demo.name}.npy"), np.load(demo / f"{demo.name}_scales.npy"))
print(demo.name, emb.shape, emb.dtype)
print("value range:", float(emb.min()), "to", float(emb.max()), "| NaN:", int(np.isnan(emb).sum()))

flat = emb.reshape(-1, emb.shape[-1])
rng = np.random.default_rng(0)
sample = flat[rng.choice(flat.shape[0], 20_000, replace=False)]
mean = sample.mean(axis=0)
_, _, vt = np.linalg.svd(sample - mean, full_matrices=False)
pcs = ((flat - mean) @ vt[:3].T).reshape(*emb.shape[:2], 3)
lo, hi = np.percentile(pcs, [2, 98], axis=(0, 1))
rgb = np.clip((pcs - lo) / (hi - lo), 0, 1)

plt.figure(figsize=(6, 7))
plt.imshow(rgb)
plt.title(f"{demo.name} — embedding PCs 1–3 as RGB")
plt.axis("off");

## Summary — satellite-specific facts

- 12 tiles downloaded, all at 10 m pixel size, CRS EPSG:32629 (UTM 29N);
  embedding and landmask shapes match for every tile.
- All downloaded tiles report 100% land pixels (no ocean/nodata masked out
  in this pilot area).
- Only year 2024 was downloaded; TESSERA itself covers 2017–2025. The
  proposed temporal design uses 2023 as the reference year (see
  `docs/research_questions.md`, Temporal Design), so 2023 tiles would
  need to be downloaded if it is adopted.
- Dequantised embedding values form a continuous range (see the printed
  min/max above), and the PCA false-colour view looks spatially coherent,
  not noise — a basic sanity check that the downloaded tiles are usable.

Open questions and missing data are tracked in `docs/` and
`data/README.md`, not duplicated here.